In [0]:
deliveries = spark.read.table('data.ipldata.deliveries')
matches = spark.read.table('data.ipldata.matches')
players = spark.read.table('data.ipldata.players')
seasons = spark.read.table('data.ipldata.seasons')
import pyspark.sql.functions as f
from pyspark.sql.window import Window

In [0]:
# Find batsman with highest percentage of team runs in a match
batsman_runs = deliveries.groupBy("match_id", "striker") \
    .agg(f.sum("batsman_runs").alias("runs"))

team_runs = deliveries.groupBy("match_id") \
    .agg(f.sum("total_runs").alias("team_total"))

df = batsman_runs.join(team_runs, "match_id") \
    .withColumn("per", (f.col("runs") * 100) / f.col("team_total"))
w = Window.partitionBy("match_id").orderBy(f.col("per").desc())

result = df.withColumn("rank", f.rank().over(w)) \
    .filter(f.col("rank") == 1)

display(result)

In [0]:
# Find matches where a single batsman scored more than 50% of team total. 
batsman_runs = deliveries.groupBy("match_id", "striker") \
    .agg(f.sum("batsman_runs").alias("runs"))

team_runs = deliveries.groupBy("match_id") \
    .agg(f.sum("total_runs").alias("team_total"))

per = batsman_runs.join(team_runs, "match_id") \
    .withColumn("per", f.round(f.col("runs") * 100 / f.col("team_total"),2))\
    .filter(f.col("per") > 50).select("match_id").distinct()

display(per)

In [0]:
# Calculate average partnership runs per team per season. 
w = Window.partitionBy("match_id", "innings").orderBy("over", "ball")
df = deliveries.withColumn(
    "wicket_flag",
    f.when(f.col("is_wicket") == True, 1).otherwise(0)
)
df = df.withColumn(
    "partnership_id",
    f.sum("wicket_flag").over(w)
)
partnership = df.groupBy("match_id", "innings", "batting_team", "partnership_id") \
    .agg(f.sum("total_runs").alias("partnership_runs"))

avg_partnership = partnership.join(matches, "match_id")\
    .groupBy("season", "batting_team") \
    .agg(f.round(f.avg("partnership_runs"), 2).alias("avg_partnership"))\
    .orderBy("season", "batting_team")

display(avg_partnership)

In [0]:
# Bowlers who never conceded a six in a season
bowler_sixes = deliveries.filter(f.col("batsman_runs") == 6) \
    .join(matches.select("match_id", "season"), "match_id") \
    .select("season", "bowler").distinct()

all_bowlers = deliveries.join(matches.select("match_id", "season"), "match_id") \
    .select("season", "bowler").distinct()

never_conceded_six = all_bowlers.join(bowler_sixes, ["season", "bowler"], "left_anti") \
    .orderBy("season", "bowler")

display(never_conceded_six)

In [0]:
# Batsmen who faced >1000 balls but never scored a century
balls_faced = deliveries.groupBy("striker") \
    .agg(f.count("*").alias("balls_faced"))

centuries = deliveries.groupBy("match_id", "striker") \
    .agg(f.sum("batsman_runs").alias("runs")) \
    .filter(f.col("runs") >= 100) \
    .select("striker").distinct()

batsmen_1000_balls = balls_faced.filter(f.col("balls_faced") > 1000) \
    .select("striker")

never_century = batsmen_1000_balls.join(centuries, "striker", "left_anti") \
    .orderBy("striker")

display(never_century)

In [0]:
# Team with highest average powerplay score (overs 1–6)
powerplay = deliveries.filter((f.col("over") >= 1) & (f.col("over") <= 6)) \
    .groupBy("batting_team", "match_id", "innings") \
    .agg(f.sum("total_runs").alias("powerplay_runs"))

avg_powerplay = powerplay.groupBy("batting_team") \
    .agg(f.round(f.avg("powerplay_runs"), 2).alias("avg_powerplay")) \
    .orderBy(f.col("avg_powerplay").desc())

display(avg_powerplay)

In [0]:
# Matches where both teams scored more than 200 runs
innings_runs = deliveries.groupBy("match_id", "innings") \
    .agg(f.sum("total_runs").alias("innings_total"))

matches = innings_runs.filter(f.col("innings_total") > 200) \
    .groupBy("match_id") \
    .agg(f.count("*").alias("teams")) \
    .filter(f.col("teams") == 2) \
    .select("match_id")

display(matches)

In [0]:
# Teams that successfully defended totals under 150
first_innings = deliveries.groupBy("match_id", "innings", "batting_team") \
    .agg(f.sum("total_runs").alias("total"))

first_innings_150 = first_innings.filter((f.col("innings") == 1) & (f.col("total") < 150)) \
    .select("match_id", "batting_team", "total")

second_innings = first_innings.filter(f.col("innings") == 2) \
    .select("match_id", "batting_team", "total") \
    .withColumnRenamed("batting_team", "chasing_team") \
    .withColumnRenamed("total", "chasing_total")

defended = first_innings_150.join(second_innings, "match_id") \
    .filter(f.col("total") > f.col("chasing_total")) \
    .select(f.col("batting_team").alias("defending_team"), "match_id", f.col("total").alias("defended_total"))

display(defended)